# Next-Token Prediction

## Learning Objectives

In the previous notebook, we converted text into:

```text
tokens
↓
token IDs
↓
embeddings
↓
context windows
↓
context embeddings
```

A context such as:

```text
["the", "cat", "sat"]
```

was represented as a tensor with shape:

```text
[T, C]
```

and a batch of contexts had shape:

```text
[B, T, C]
```

where:

```text
B = batch size / number of examples
T = context length
C = embedding dimension
```

But an embedding tensor by itself does not predict anything.

In this notebook, we will turn context representations into an actual language-model training objective.

We will build the following pipeline:

```text
context token IDs
[B, T]

↓

embedding lookup

[B, T, C]

↓

combine context information

[B, T*C]

↓

prediction layer

[B, V]

↓

logits over vocabulary

↓

cross-entropy loss

↓

backpropagation

↓

parameter updates
```

where:

```text
V = vocabulary size
```

By the end of this notebook, we should understand:

- why a next-token model needs one score for every vocabulary token
- what a logit is
- why logits are not probabilities
- how softmax converts logits into probabilities
- why next-token prediction is a classification problem
- how cross-entropy compares predictions with the correct token
- why `CrossEntropyLoss` expects logits rather than probabilities
- how gradients reach both the prediction layer and the embedding table
- how next-token training changes embeddings
- why the same context can legitimately have multiple possible next tokens

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.1


 ## 1. Recreate the Tiny Corpus

This notebook is intentionally self-contained.

We will reuse the same tiny corpus:

```text
the cat sat on the mat
the cat ate the fish
the dog sat on the rug
```

We use a tiny dataset because every training example can be inspected manually.

The purpose is not to build a useful language model yet.

The purpose is to understand exactly what next-token training is doing.

In [2]:
sentences = [
    "the cat sat on the mat",
    "the cat ate the fish",
    "the dog sat on the rug",
]

tokenized_sentences = [
    sentence.split()
    for sentence in sentences
]

all_tokens = [
    token
    for sentence_tokens in tokenized_sentences
    for token in sentence_tokens
]

vocab = sorted(set(all_tokens))

token_to_id = {
    token: idx
    for idx, token in enumerate(vocab)
}

id_to_token = {
    idx: token
    for token, idx in token_to_id.items()
}

print("Vocabulary:", vocab)
print("Vocabulary size:", len(vocab))
print("Token → ID:", token_to_id)

Vocabulary: ['ate', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']
Vocabulary size: 9
Token → ID: {'ate': 0, 'cat': 1, 'dog': 2, 'fish': 3, 'mat': 4, 'on': 5, 'rug': 6, 'sat': 7, 'the': 8}


## 2. Encode the Sentences

The vocabulary IDs are categorical indexes.

We convert each tokenized sentence into a tensor of integer IDs so that PyTorch can use those IDs for embedding lookup.

The transformation is:

```text
words
↓
vocabulary lookup
↓
integer token IDs
```

We continue to keep sentences separate so that our context windows do not accidentally cross sentence boundaries.

In [3]:
encoded_sentences = [
    torch.tensor(
        [token_to_id[token] for token in sentence_tokens],
        dtype=torch.long,
    )
    for sentence_tokens in tokenized_sentences
]

for sentence, encoded in zip(sentences, encoded_sentences):
    print(sentence)
    print(encoded)
    print()

the cat sat on the mat
tensor([8, 1, 7, 5, 8, 4])

the cat ate the fish
tensor([8, 1, 0, 8, 3])

the dog sat on the rug
tensor([8, 2, 7, 5, 8, 6])



## 3. Build Context → Target Examples

We again use:

```text
context length T = 3
```

A training example therefore looks like:

```text
three previous tokens → next token
```

For example:

```text
the cat sat → on
```

Numerically:

```text
[8, 1, 7] → 5
```

The target is not a continuous value.

It is the ID of one token from our vocabulary.

That gives us the first important connection:

> Next-token prediction can be formulated as a classification problem.

If the vocabulary contains `V` tokens, then every prediction chooses among `V` possible classes.

In [4]:
context_length = 3

contexts = []
targets = []

for sentence_ids in encoded_sentences:
    for i in range(len(sentence_ids) - context_length):
        contexts.append(
            sentence_ids[i:i + context_length]
        )
        targets.append(
            sentence_ids[i + context_length]
        )

contexts = torch.stack(contexts)
targets = torch.stack(targets)

print("Contexts shape:", contexts.shape)
print("Targets shape:", targets.shape)

Contexts shape: torch.Size([8, 3])
Targets shape: torch.Size([8])


In [5]:
for context_ids, target_id in zip(contexts, targets):
    context_tokens = [
        id_to_token[token_id.item()]
        for token_id in context_ids
    ]

    target_token = id_to_token[target_id.item()]

    print(context_tokens, "→", target_token)

['the', 'cat', 'sat'] → on
['cat', 'sat', 'on'] → the
['sat', 'on', 'the'] → mat
['the', 'cat', 'ate'] → the
['cat', 'ate', 'the'] → fish
['the', 'dog', 'sat'] → on
['dog', 'sat', 'on'] → the
['sat', 'on', 'the'] → rug


## 4. One Context Can Have Multiple Valid Next Tokens

Our tiny corpus already contains an important language-modeling phenomenon.

We have:

```text
["sat", "on", "the"] → "mat"
```

and also:

```text
["sat", "on", "the"] → "rug"
```

The exact same context has two different observed next tokens.

This is not an error in the dataset.

Language is not generally a deterministic mapping:

```text
context → exactly one possible next token
```

A context such as:

```text
I went to the
```

could plausibly continue with:

```text
store
park
office
airport
restaurant
...
```

A language model therefore does not fundamentally need to produce only one hard answer.

Instead, it learns a distribution over possible next tokens:

```text
P(next token | context)
```

For our vocabulary of size `V = 9`, the model will eventually assign a value to every possible token:

```text
P(ate  | context)
P(cat  | context)
P(dog  | context)
P(fish | context)
P(mat  | context)
P(on   | context)
P(rug  | context)
P(sat  | context)
P(the  | context)
```

The model can then assign substantial probability to more than one plausible continuation.

This probabilistic view is central to language modeling.

## 5. Create the Embedding Table

As before, token IDs are only indexes.

We convert each token ID into a learnable continuous representation.

We will again use:

```text
embedding dimension C = 4
```

This remains a deliberately tiny teaching value.

Our embedding table therefore has shape:

```text
[V, C]
```

With:

```text
V = 9
C = 4
```

we expect:

```text
[9, 4]
```

Unlike the previous notebook, however, these embeddings will eventually participate in a real prediction loss.

That means they will finally receive gradients and be updated during training.

In [6]:
vocab_size = len(vocab)
embedding_dim = 4

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim,
)

context_embeddings = embedding(contexts)

print("Embedding table shape:", embedding.weight.shape)
print("Contexts shape:", contexts.shape)
print("Context embeddings shape:", context_embeddings.shape)

Embedding table shape: torch.Size([9, 4])
Contexts shape: torch.Size([8, 3])
Context embeddings shape: torch.Size([8, 3, 4])


## 6. From Many Token Representations to One Prediction

For each training example, we currently have:

```text
T = 3 token positions
```

and each position has:

```text
C = 4 features
```

So one context has shape:

```text
[T, C]
=
[3, 4]
```

Conceptually:

```text
"the" → [• • • •]
"cat" → [• • • •]
"sat" → [• • • •]
```

But our training example has only **one target**:

```text
"on"
```

So we need some mechanism that uses all three context-token representations to produce one next-token prediction.

There are many possible architectures for doing this.

Later we will use attention and Transformers.

For now, we deliberately choose the simplest possible mechanism:

> Concatenate the embeddings of all context positions into one long vector.

With:

```text
T = 3
C = 4
```

we have:

```text
[T, C]
=
[3, 4]
```

and flatten it into:

```text
[T × C]
=
[12]
```

Conceptually:

```text
token 1 embedding [4]
+
token 2 embedding [4]
+
token 3 embedding [4]

↓

one context vector [12]
```

For an entire batch:

```text
[B, T, C]
↓
flatten T and C
↓
[B, T*C]
```

This is not how a Transformer combines context.

It is simply the smallest architecture that lets us study the language-model objective before introducing more sophisticated sequence processing.

In [7]:
flattened_contexts = context_embeddings.reshape(
    context_embeddings.shape[0],
    -1,
)

print(
    "Before flattening:",
    context_embeddings.shape,
)

print(
    "After flattening:",
    flattened_contexts.shape,
)

Before flattening: torch.Size([8, 3, 4])
After flattening: torch.Size([8, 12])


## 7. What Did `reshape` Actually Do?

No information was mathematically averaged or combined yet.

We simply rearranged the dimensions.

One example originally looked like:

```text
[
    [a1, a2, a3, a4],   # token position 1
    [b1, b2, b3, b4],   # token position 2
    [c1, c2, c3, c4],   # token position 3
]
```

with shape:

```text
[3, 4]
```

Flattening produces:

```text
[
    a1, a2, a3, a4,
    b1, b2, b3, b4,
    c1, c2, c3, c4
]
```

with shape:

```text
[12]
```

The ordering still preserves position implicitly:

```text
first 4 values
→ first token

next 4 values
→ second token

last 4 values
→ third token
```

This is important because:

```text
["the", "cat", "sat"]
```

should not necessarily mean the same thing as:

```text
["sat", "cat", "the"]
```

Flattening preserves that ordering because the embeddings occupy different sections of the resulting vector.

## 8. Predicting Over the Vocabulary

Our vocabulary contains:

```text
V = 9
```

possible tokens.

For every context, the model therefore needs to produce **9 scores**:

```text
score for "ate"
score for "cat"
score for "dog"
score for "fish"
score for "mat"
score for "on"
score for "rug"
score for "sat"
score for "the"
```

So our model must transform:

```text
context representation
[T*C]
```

into:

```text
vocabulary scores
[V]
```

For a batch:

```text
[B, T*C]
↓
prediction layer
↓
[B, V]
```

The raw output scores are called **logits**.

A logit is simply an unrestricted real-valued score.

For example:

```text
ate   → -0.7
cat   →  1.2
dog   → -0.1
fish  →  0.4
mat   →  2.1
on    →  3.4
rug   →  1.8
sat   → -0.3
the   →  0.9
```

These values are **not probabilities**.

They:

- can be negative
- can be greater than 1
- do not need to sum to 1

Higher logits indicate that the model currently favors those tokens more strongly.

## 9. A Linear Layer Can Produce Vocabulary Logits

Our flattened context has:

```text
T * C
```

features.

We want:

```text
V
```

output scores.

A linear layer can perform exactly this transformation:

```text
Linear(T*C, V)
```

In our experiment:

```text
T = 3
C = 4
V = 9
```

therefore:

```text
T*C = 12
```

and the prediction layer is:

```python
nn.Linear(12, 9)
```

Its job is:

```text
12-dimensional context representation
↓
9 vocabulary logits
```

This is our first extremely small next-token prediction model.

In [8]:
input_size = context_length * embedding_dim

prediction_layer = nn.Linear(
    input_size,
    vocab_size,
)

logits = prediction_layer(flattened_contexts)

print("Flattened contexts shape:", flattened_contexts.shape)
print("Prediction weight shape:", prediction_layer.weight.shape)
print("Prediction bias shape:", prediction_layer.bias.shape)
print("Logits shape:", logits.shape)

Flattened contexts shape: torch.Size([8, 12])
Prediction weight shape: torch.Size([9, 12])
Prediction bias shape: torch.Size([9])
Logits shape: torch.Size([8, 9])


## 10. Understanding `[B, V]`

The output tensor has shape:

```text
[B, V]
```

where:

```text
B = number of contexts
V = number of vocabulary tokens
```

Each row corresponds to one training example.

Each column corresponds to one vocabulary token.

Conceptually:

```text
                   vocabulary token
          ───────────────────────────────►

context 0    [ •  •  •  •  •  •  •  •  • ]
context 1    [ •  •  •  •  •  •  •  •  • ]
context 2    [ •  •  •  •  •  •  •  •  • ]
...
```

For one context:

```text
["the", "cat", "sat"]
```

the model produces one score for every possible next token.

At this point the parameters are still random, so the scores have no learned meaning.

But structurally we now have what a next-token model needs:

```text
context
↓
V competing scores
```

In [9]:
example_index = 0

print("Context:")
print([
    id_to_token[token_id.item()]
    for token_id in contexts[example_index]
])

print("\nCorrect target:")
print(
    id_to_token[
        targets[example_index].item()
    ]
)

print("\nLogits:")
for token_id, logit in enumerate(logits[example_index]):
    print(
        f"{id_to_token[token_id]:>5}: "
        f"{logit.item(): .4f}"
    )

Context:
['the', 'cat', 'sat']

Correct target:
on

Logits:
  ate:  0.9486
  cat: -0.9575
  dog:  0.8255
 fish: -0.2829
  mat:  0.0059
   on: -0.1200
  rug:  0.4439
  sat:  0.7850
  the: -0.2385


## 11. Logits Are Not Probabilities

The model's output is currently a vector of raw scores:

```text
z = [z₁, z₂, ..., zᵥ]
```

These scores are called logits.

They can look like:

```text
[-0.2, 1.4, 0.7, -1.1, ...]
```

But probabilities should satisfy two properties:

```text
each probability is between 0 and 1

and

all probabilities sum to 1
```

Logits do not satisfy those constraints.

To convert logits into a probability distribution, we can use **softmax**.

For logit `z_i`:

$$
P_i =
\frac{e^{z_i}}
{\sum_j e^{z_j}}
$$

Softmax does two things:

1. exponentiation makes every value positive
2. division by the total makes all values sum to 1

The relative ordering is preserved:

```text
larger logit
→ larger probability
```

but the result is now a valid categorical probability distribution.

In [10]:
probabilities = torch.softmax(
    logits,
    dim=-1,
)

print("Probability shape:", probabilities.shape)

print(
    "Probability sum for first example:",
    probabilities[0].sum().item(),
)

Probability shape: torch.Size([8, 9])
Probability sum for first example: 1.0


In [11]:
print("Context:")
print([
    id_to_token[token_id.item()]
    for token_id in contexts[0]
])

print("\nCorrect target:")
print(id_to_token[targets[0].item()])

print("\nPredicted probabilities:")

for token_id, probability in enumerate(probabilities[0]):
    print(
        f"{id_to_token[token_id]:>5}: "
        f"{probability.item():.4f}"
    )

Context:
['the', 'cat', 'sat']

Correct target:
on

Predicted probabilities:
  ate: 0.2077
  cat: 0.0309
  dog: 0.1836
 fish: 0.0606
  mat: 0.0809
   on: 0.0713
  rug: 0.1254
  sat: 0.1763
  the: 0.0634


## 12. Interpreting the Probability Distribution

For one context, softmax produces:

```text
V probabilities
```

that sum to:

```text
1
```

We can interpret them as the model's current distribution over the next token:

```text
P(next token = "ate"  | context)
P(next token = "cat"  | context)
P(next token = "dog"  | context)
...
```

At this stage, however, the probabilities are effectively random because the embedding table and prediction layer were randomly initialized.

We now need a way to tell the model:

> The correct next token for this training example was `"on"`.

and quantify how bad its current prediction was.

That quantity will be the **cross-entropy loss**.

## 13. From Probability to Loss

For the first training example, the correct target is:

```text
"on"
```

The model currently assigns some probability to `"on"`.

Call that probability:

$$
p_{\text{correct}}
$$

A simple question is:

> How should we turn the probability assigned to the correct answer into a loss?

We want the loss to behave like this:

```text
high probability for correct token
→ small loss

low probability for correct token
→ large loss
```

The standard choice for classification is the **negative log-likelihood**:

$$
L = -\log(p_{\text{correct}})
$$

Let's inspect what this function does.

If the model assigns:

```text
p(correct) = 1.0
```

then:

$$
-\log(1)=0
$$

so the loss is zero.

If:

```text
p(correct) = 0.5
```

then:

$$
-\log(0.5)\approx0.693
$$

If:

```text
p(correct) = 0.1
```

then:

$$
-\log(0.1)\approx2.303
$$

If:

```text
p(correct) = 0.01
```

then:

$$
-\log(0.01)\approx4.605
$$

So:

```text
correct token receives high probability
→ low loss

correct token receives low probability
→ high loss
```

This gives optimization exactly the direction we want.

In [12]:
test_probabilities = torch.tensor([
    1.0,
    0.9,
    0.5,
    0.1,
    0.01,
])

test_losses = -torch.log(test_probabilities)

for probability, loss in zip(
    test_probabilities,
    test_losses,
):
    print(
        f"p(correct) = {probability.item():.2f}"
        f"  →  loss = {loss.item():.4f}"
    )

p(correct) = 1.00  →  loss = -0.0000
p(correct) = 0.90  →  loss = 0.1054
p(correct) = 0.50  →  loss = 0.6931
p(correct) = 0.10  →  loss = 2.3026
p(correct) = 0.01  →  loss = 4.6052


## 14. Calculate the Loss for Our First Prediction Manually

Our first context is:

```text
["the", "cat", "sat"]
```

and the correct next token is:

```text
"on"
```

The model produced a probability distribution over all `V = 9` tokens.

To compute the loss for this example, we do not need all nine probabilities directly.

We need the probability assigned to the observed target:

```text
P("on" | "the cat sat")
```

Then:

$$
L =
-\log
P(\text{"on"} \mid \text{"the cat sat"})
$$

This is the negative log-likelihood of the observed next token.

Our model is currently randomly initialized, so we expect the loss to be relatively large.

In [13]:
example_index = 0

target_id = targets[example_index]

correct_probability = probabilities[
    example_index,
    target_id
]

manual_loss = -torch.log(correct_probability)

print(
    "Context:",
    [
        id_to_token[token_id.item()]
        for token_id in contexts[example_index]
    ],
)

print(
    "Correct target:",
    id_to_token[target_id.item()],
)

print(
    "Probability assigned to correct target:",
    correct_probability.item(),
)

print(
    "Manual negative log-likelihood:",
    manual_loss.item(),
)

Context: ['the', 'cat', 'sat']
Correct target: on
Probability assigned to correct target: 0.07132434844970703
Manual negative log-likelihood: 2.640517473220825


## 15. Why Use the Log?

At first, we might wonder why we do not simply use:

```text
loss = 1 - p(correct)
```

Negative log-likelihood has several useful properties.

### 1. It strongly penalizes confident mistakes

Compare:

```text
p(correct) = 0.9
```

with:

```text
p(correct) = 0.01
```

The second prediction is not merely somewhat worse.

The model was extremely confident in alternatives while assigning almost no probability to the observed answer.

The logarithm penalizes that strongly.

### 2. Probabilities across a sequence multiply

Suppose a model assigns probabilities to a sequence of observed tokens:

$$
p_1,p_2,p_3,\ldots,p_N
$$

Under the autoregressive factorization we will study later, sequence likelihood involves multiplying probabilities:

$$
P(\text{sequence})
=
\prod_{i=1}^{N}p_i
$$

Taking the logarithm turns products into sums:

$$
\log
\left(
\prod_i p_i
\right)
=
\sum_i \log p_i
$$

Sums are much easier and numerically safer to optimize than multiplying many tiny probabilities.

### 3. It connects maximum likelihood to loss minimization

Training can be viewed as trying to make the observed data highly probable.

Maximizing:

$$
\log P(\text{data})
$$

is equivalent to minimizing:

$$
-\log P(\text{data})
$$

This connection between:

```text
probability
→ likelihood
→ log-likelihood
→ cross-entropy loss
```

is fundamental to language-model training.

## 16. Cross-Entropy for One Correct Class

Our target is represented as one correct class ID:

```text
target = token ID for "on"
```

The model produces a probability distribution:

$$
p_1,p_2,\ldots,p_V
$$

over all vocabulary tokens.

We can imagine the true target distribution as a **one-hot vector**.

For example, if `"on"` were vocabulary ID 5:

```text
target distribution

ate   0
cat   0
dog   0
fish  0
mat   0
on    1
rug   0
sat   0
the   0
```

Cross-entropy between the target distribution `y` and model distribution `p` is:

$$
H(y,p)
=
-\sum_i y_i\log p_i
$$

But every element of `y` is zero except the correct class.

Therefore all terms disappear except one:

$$
H(y,p)
=
-\log p_{\text{correct}}
$$

So for ordinary single-label classification:

```text
cross-entropy
=
negative log-probability of the correct class
```

This is why next-token prediction naturally uses cross-entropy loss.

In [14]:
target_distribution = torch.zeros(vocab_size)

target_distribution[target_id] = 1.0

manual_cross_entropy = -torch.sum(
    target_distribution
    * torch.log(probabilities[example_index])
)

print("Target ID:", target_id.item())
print("Target token:", id_to_token[target_id.item()])

print("\nOne-hot target distribution:")
print(target_distribution)

print(
    "\nNegative log-likelihood:",
    manual_loss.item(),
)

print(
    "Cross-entropy from one-hot target:",
    manual_cross_entropy.item(),
)

Target ID: 5
Target token: on

One-hot target distribution:
tensor([0., 0., 0., 0., 0., 1., 0., 0., 0.])

Negative log-likelihood: 2.640517473220825
Cross-entropy from one-hot target: 2.640517473220825


## 17. PyTorch's `CrossEntropyLoss`

We could manually perform:

```text
logits
↓
softmax
↓
select correct probability
↓
-log()
```

But PyTorch provides:

```python
nn.CrossEntropyLoss()
```

There is an important API detail:

> `CrossEntropyLoss` expects raw logits, not softmax probabilities.

So we should do:

```python
loss = criterion(logits, targets)
```

not:

```python
loss = criterion(probabilities, targets)
```

Conceptually, PyTorch combines the relevant operations internally:

```text
logits
↓
log-softmax
↓
negative log-likelihood
↓
cross-entropy
```

Doing these operations together is also more numerically stable than explicitly computing probabilities first.

The expected shapes for our problem are:

```text
logits  [B, V]
targets [B]
```

Each row of logits contains `V` class scores.

Each target contains the integer ID of the correct class.

In [15]:
criterion = nn.CrossEntropyLoss()

pytorch_single_loss = criterion(
    logits[0:1],
    targets[0:1],
)

print(
    "Manual loss:",
    manual_loss.item(),
)

print(
    "PyTorch CrossEntropyLoss:",
    pytorch_single_loss.item(),
)

print(
    "Difference:",
    abs(
        manual_loss.item()
        - pytorch_single_loss.item()
    ),
)

Manual loss: 2.640517473220825
PyTorch CrossEntropyLoss: 2.6405177116394043
Difference: 2.384185791015625e-07


## 18. Loss Across the Entire Batch

So far we examined one example.

But we have:

```text
B = 8
```

training examples.

For every example, the model assigns a probability to the correct target.

PyTorch's default `CrossEntropyLoss` computes the loss for each example and then takes the **mean** across the batch.

Conceptually:

$$
L_{\text{batch}}
=
\frac{1}{B}
\sum_{b=1}^{B}
-\log
p_b(y_b)
$$

where:

```text
y_b
```

is the correct next-token ID for example `b`.

This gives us one scalar loss for the entire batch.

That scalar is what we will eventually call:

```python
loss.backward()
```

on.

In [16]:
batch_loss = criterion(
    logits,
    targets,
)

print("Batch cross-entropy loss:")
print(batch_loss.item())

Batch cross-entropy loss:
2.4866445064544678


## 19. What Loss Would Uniform Random Guessing Produce?

Our vocabulary contains:

```text
V = 9
```

tokens.

If a model assigned every token equal probability:

$$
p_i=\frac{1}{V}
$$

then the probability assigned to the correct token would be:

$$
\frac{1}{9}
$$

The corresponding cross-entropy is:

$$
-\log\left(\frac{1}{V}\right)
=
\log(V)
$$

For our vocabulary:

$$
\log(9)
$$

gives us a useful reference point.

This does **not** mean every randomly initialized neural network must produce exactly this loss.

Random initialization does not generally create exactly uniform logits.

But `log(V)` gives us a simple baseline for what uniform uncertainty looks like.

In [17]:
uniform_baseline_loss = torch.log(
    torch.tensor(
        float(vocab_size)
    )
)

print(
    "Uniform probability per token:",
    1 / vocab_size,
)

print(
    "Uniform baseline cross-entropy:",
    uniform_baseline_loss.item(),
)

print(
    "Current model batch loss:",
    batch_loss.item(),
)

Uniform probability per token: 0.1111111111111111
Uniform baseline cross-entropy: 2.1972246170043945
Current model batch loss: 2.4866445064544678


## 20. What Should Receive Gradients?

Our prediction currently follows this computation graph:

```text
token IDs
↓
embedding table
↓
context embeddings
↓
flatten
↓
prediction layer
↓
logits
↓
cross-entropy loss
```

Two components contain learnable parameters:

```text
1. embedding.weight
2. prediction_layer.weight and prediction_layer.bias
```

Therefore, when we call:

```python
batch_loss.backward()
```

the chain rule should propagate gradients backward through the entire graph.

The prediction layer receives gradients because its weights directly produced the logits.

But importantly, the embedding table should also receive gradients.

This means the training objective can change how tokens themselves are represented.

Before calling `backward()`, their `.grad` fields should currently be:

```text
None
```

After `backward()`, they should contain tensors of gradients.

In [18]:
print("Before backward:")

print(
    "Embedding gradient:",
    embedding.weight.grad,
)

print(
    "Prediction weight gradient:",
    prediction_layer.weight.grad,
)

print(
    "Prediction bias gradient:",
    prediction_layer.bias.grad,
)

Before backward:
Embedding gradient: None
Prediction weight gradient: None
Prediction bias gradient: None


In [19]:
batch_loss.backward()

In [20]:
print("After backward:")

print(
    "Embedding gradient shape:",
    embedding.weight.grad.shape,
)

print(
    "Embedding mean |gradient|:",
    embedding.weight.grad.abs().mean().item(),
)

print(
    "\nPrediction weight gradient shape:",
    prediction_layer.weight.grad.shape,
)

print(
    "Prediction weight mean |gradient|:",
    prediction_layer.weight.grad.abs().mean().item(),
)

print(
    "\nPrediction bias gradient shape:",
    prediction_layer.bias.grad.shape,
)

print(
    "Prediction bias mean |gradient|:",
    prediction_layer.bias.grad.abs().mean().item(),
)

After backward:
Embedding gradient shape: torch.Size([9, 4])
Embedding mean |gradient|: 0.02776317298412323

Prediction weight gradient shape: torch.Size([9, 12])
Prediction weight mean |gradient|: 0.11478468775749207

Prediction bias gradient shape: torch.Size([9])
Prediction bias mean |gradient|: 0.1174585148692131


## 21. Why the Embeddings Receive Gradients

This is the first time our embeddings have an actual learning signal.

The loss depends on:

```text
logits
```

which depend on:

```text
flattened context embeddings
```

which depend on:

```text
embedding.weight
```

Therefore, by the chain rule:

$$
\frac{\partial L}
{\partial \text{embedding.weight}}
\neq 0
$$

for embedding rows that influenced the loss.

Conceptually:

```text
bad next-token prediction
↓
loss
↓
gradient through prediction layer
↓
gradient through context representation
↓
gradient into token embeddings
```

This answers an important question from Notebook 1:

> How do embedding vectors acquire useful values?

They are not assigned semantic meaning directly.

They are updated because changing them can help reduce the model's training objective.

Later, with large amounts of language data, tokens that occur in related predictive contexts may develop useful geometric relationships.

But those relationships are an **emergent consequence of optimization**, not something manually encoded into the embedding table.

## 22. A Gradient Is Not Yet an Update

After calling:

```python
batch_loss.backward()
```

PyTorch calculated gradients for the learnable parameters.

But the parameters themselves have **not changed yet**.

Backpropagation answers:

```text
How would changing each parameter affect the loss?
```

It does not perform the change.

To actually learn, we still need an optimization step:

```text
forward pass
↓
loss
↓
backward()
↓
gradients
↓
optimizer.step()
↓
updated parameters
```

We will inspect one embedding before and after the update so that the parameter change is directly observable.

We will use the embedding for:

```text
"cat"
```

as our example.

In [21]:
cat_id = token_to_id["cat"]

cat_embedding_before = (
    embedding.weight[cat_id]
    .detach()
    .clone()
)

print("cat embedding before update:")
print(cat_embedding_before)

print("\ncat gradient:")
print(embedding.weight.grad[cat_id])

cat embedding before update:
tensor([ 0.6784, -1.2345, -0.0431, -1.6047])

cat gradient:
tensor([-0.0332,  0.0176,  0.0030,  0.0129])


## 23. Perform One Optimization Step

We will use stochastic gradient descent:

```python
torch.optim.SGD
```

with a deliberately simple learning rate.

For a parameter $\theta$, one SGD update is approximately:

$$
\theta_{\text{new}}
=
\theta_{\text{old}}
-
\eta
\frac{\partial L}{\partial \theta}
$$

where:

```text
η = learning rate
```

We will use:

```text
learning rate = 0.1
```

for this small experiment.

The important point is that we have **already calculated the gradients** using:

```python
batch_loss.backward()
```

So the optimizer can now use those stored `.grad` values to modify the parameters.

In [22]:
optimizer = torch.optim.SGD(
    list(embedding.parameters())
    + list(prediction_layer.parameters()),
    lr=0.1,
)

optimizer.step()

cat_embedding_after = (
    embedding.weight[cat_id]
    .detach()
    .clone()
)

print("cat embedding before:")
print(cat_embedding_before)

print("\ncat embedding after:")
print(cat_embedding_after)

print("\nchange:")
print(cat_embedding_after - cat_embedding_before)

cat embedding before:
tensor([ 0.6784, -1.2345, -0.0431, -1.6047])

cat embedding after:
tensor([ 0.6817, -1.2363, -0.0434, -1.6060])

change:
tensor([ 0.0033, -0.0018, -0.0003, -0.0013])


## 24. Verify the SGD Update Manually

For the `"cat"` embedding, SGD should have applied:

$$
e_{\text{after}}
=
e_{\text{before}}
-
\eta g
$$

where:

```text
e = cat embedding
g = cat embedding gradient
η = learning rate
```

Because we saved both the old embedding and its gradient, we can verify the optimizer's update directly.

In [23]:
cat_gradient = (
    embedding.weight.grad[cat_id]
    .detach()
    .clone()
)

expected_cat_embedding = (
    cat_embedding_before
    - 0.1 * cat_gradient
)

print("Expected from SGD equation:")
print(expected_cat_embedding)

print("\nActual after optimizer.step():")
print(cat_embedding_after)

print("\nMaximum absolute difference:")
print(
    (
        expected_cat_embedding
        - cat_embedding_after
    )
    .abs()
    .max()
    .item()
)

Expected from SGD equation:
tensor([ 0.6817, -1.2363, -0.0434, -1.6060])

Actual after optimizer.step():
tensor([ 0.6817, -1.2363, -0.0434, -1.6060])

Maximum absolute difference:
0.0


## 25. Did One Update Improve the Loss?

Changing parameters is not itself evidence that learning improved.

We should measure the objective again.

However, we cannot reuse our old `logits` tensor.

Those logits were computed using the **old parameter values**.

After `optimizer.step()`, we need a new forward pass:

```text
updated parameters
↓
new embeddings
↓
new logits
↓
new loss
```

This is why each training iteration performs the forward pass again.

In [24]:
with torch.no_grad():
    new_context_embeddings = embedding(contexts)

    new_flattened_contexts = (
        new_context_embeddings.reshape(
            new_context_embeddings.shape[0],
            -1,
        )
    )

    new_logits = prediction_layer(
        new_flattened_contexts
    )

    new_batch_loss = criterion(
        new_logits,
        targets,
    )

print("Loss before update:", batch_loss.item())
print("Loss after update:", new_batch_loss.item())
print(
    "Loss change:",
    new_batch_loss.item() - batch_loss.item(),
)

Loss before update: 2.4866445064544678
Loss after update: 2.1856489181518555
Loss change: -0.3009955883026123


## 26. Why `torch.no_grad()` for Evaluation?

When recomputing the loss after the update, we used:

```python
with torch.no_grad():
```

We only wanted to **measure** the new loss.

We were not preparing another backward pass yet.

Without `torch.no_grad()`, PyTorch would construct a new computational graph and store information needed for gradient calculation.

That would be unnecessary for this measurement.

The distinction is:

```text
training forward pass
→ need gradients
→ build computation graph

evaluation / inspection
→ no backward pass needed
→ torch.no_grad()
```

In [25]:
new_probabilities = torch.softmax(
    new_logits,
    dim=-1,
)

old_correct_probability = probabilities[
    0,
    targets[0]
].item()

new_correct_probability = new_probabilities[
    0,
    targets[0]
].item()

print(
    "First context:",
    [
        id_to_token[token_id.item()]
        for token_id in contexts[0]
    ],
)

print(
    "Target:",
    id_to_token[targets[0].item()],
)

print(
    "\nP(correct) before:",
    old_correct_probability,
)

print(
    "P(correct) after:",
    new_correct_probability,
)

First context: ['the', 'cat', 'sat']
Target: on

P(correct) before: 0.07132434844970703
P(correct) after: 0.11641550809144974


## 27. Gradients Persist Until We Clear Them

After:

```python
optimizer.step()
```

the gradients are still stored in:

```python
parameter.grad
```

The optimizer changes parameters, but it does not automatically erase their gradients.

If we perform another:

```python
loss.backward()
```

without clearing them first, PyTorch will **accumulate** the new gradients into the existing ones.

We already observed this behavior in Module 1.

The usual training loop therefore contains:

```python
optimizer.zero_grad()
```

before the next backward pass.

This is another example of the same training mechanics carrying directly into language models.

In [26]:
print(
    "Embedding mean |gradient| before zero_grad:",
    embedding.weight.grad.abs().mean().item(),
)

optimizer.zero_grad()

print(
    "Embedding gradient after zero_grad:",
    embedding.weight.grad,
)

Embedding mean |gradient| before zero_grad: 0.02776317298412323
Embedding gradient after zero_grad: None


## 28. From One Update to Training

We have now manually inspected one complete optimization step:

```text
contexts [B, T]
↓
embedding lookup
[B, T, C]
↓
flatten
[B, T*C]
↓
linear prediction layer
[B, V]
↓
cross-entropy
scalar loss
↓
backward()
gradients
↓
optimizer.step()
updated parameters
```

Training simply repeats this process.

For every optimization step:

```text
1. clear old gradients
2. compute embeddings
3. compute context representations
4. compute logits
5. compute cross-entropy loss
6. backpropagate
7. update parameters
```

The important point is that **both**:

```text
embedding table
and
prediction layer
```

are trained jointly from the same next-token objective.

In [27]:
torch.manual_seed(42)

embedding = nn.Embedding(
    vocab_size,
    embedding_dim,
)

prediction_layer = nn.Linear(
    context_length * embedding_dim,
    vocab_size,
)

optimizer = torch.optim.SGD(
    list(embedding.parameters())
    + list(prediction_layer.parameters()),
    lr=0.1,
)

criterion = nn.CrossEntropyLoss()

## 29. The Training Loop

Now we can write the complete training loop.

Notice that nothing fundamentally new happens inside it.

The loop simply automates the sequence we already inspected manually:

```text
zero gradients
→ forward
→ loss
→ backward
→ update
```

We will train on the entire tiny dataset as one batch.

That is reasonable here because:

```text
B = 8
```

is extremely small.

Later, larger datasets will require mini-batch training.

In [28]:
num_epochs = 500

loss_history = []

for epoch in range(num_epochs):
    optimizer.zero_grad()

    context_embeddings = embedding(contexts)

    flattened_contexts = context_embeddings.reshape(
        context_embeddings.shape[0],
        -1,
    )

    logits = prediction_layer(
        flattened_contexts
    )

    loss = criterion(
        logits,
        targets,
    )

    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if epoch % 100 == 0:
        print(
            f"epoch {epoch:3d} "
            f"loss = {loss.item():.6f}"
        )

epoch   0 loss = 2.486645
epoch 100 loss = 0.206125
epoch 200 loss = 0.187567
epoch 300 loss = 0.182143
epoch 400 loss = 0.179617


## 30. What Should We Expect?

This dataset is tiny, and our model has enough parameters to memorize much of it.

So we expect training loss to fall dramatically.

But there is one unavoidable ambiguity:

```text
["sat", "on", "the"] → "mat"
```

and:

```text
["sat", "on", "the"] → "rug"
```

The model receives the **same input context** for both examples.

Our architecture is deterministic before sampling:

```text
same context
→ same logits
→ same probability distribution
```

It cannot produce one probability distribution for the first occurrence and a different distribution for the second when the inputs are identical.

Therefore it cannot assign:

```text
100% probability to "mat"
```

and simultaneously:

```text
100% probability to "rug"
```

for the same context.

The best solution for that context must distribute probability across the observed alternatives.

This means even on our tiny training set, the theoretically optimal cross-entropy does not have to be exactly zero.

That is not an optimization failure.

It follows from uncertainty in the training data given the information available to the model.

In [29]:
print("Initial loss:", loss_history[0])
print("Final loss:", loss_history[-1])

Initial loss: 2.4866445064544678
Final loss: 0.17818431556224823


In [30]:
with torch.no_grad():
    context_embeddings = embedding(contexts)

    flattened_contexts = context_embeddings.reshape(
        context_embeddings.shape[0],
        -1,
    )

    logits = prediction_layer(
        flattened_contexts
    )

    probabilities = torch.softmax(
        logits,
        dim=-1,
    )

for i in range(len(contexts)):
    context_tokens = [
        id_to_token[token_id.item()]
        for token_id in contexts[i]
    ]

    target_token = id_to_token[
        targets[i].item()
    ]

    predicted_id = probabilities[i].argmax().item()
    predicted_token = id_to_token[predicted_id]

    target_probability = probabilities[
        i,
        targets[i]
    ].item()

    print(
        f"{context_tokens}"
        f" → target={target_token:>4}"
        f" | prediction={predicted_token:>4}"
        f" | P(target)={target_probability:.4f}"
    )

['the', 'cat', 'sat'] → target=  on | prediction=  on | P(target)=0.9959
['cat', 'sat', 'on'] → target= the | prediction= the | P(target)=0.9955
['sat', 'on', 'the'] → target= mat | prediction= mat | P(target)=0.4970
['the', 'cat', 'ate'] → target= the | prediction= the | P(target)=0.9941
['cat', 'ate', 'the'] → target=fish | prediction=fish | P(target)=0.9936
['the', 'dog', 'sat'] → target=  on | prediction=  on | P(target)=0.9967
['dog', 'sat', 'on'] → target= the | prediction= the | P(target)=0.9974
['sat', 'on', 'the'] → target= rug | prediction= mat | P(target)=0.4970


## 31. Interpreting the Final Training Result

The model reduced the training loss from approximately:

```text
2.4866
```

to:

```text
0.1782
```

Most contexts became almost perfectly predictable:

```text
["the", "cat", "sat"] → "on"    P(target) ≈ 0.996
["cat", "sat", "on"]  → "the"   P(target) ≈ 0.996
["the", "cat", "ate"] → "the"   P(target) ≈ 0.994
```

But one context occurs twice with different targets:

```text
["sat", "on", "the"] → "mat"
["sat", "on", "the"] → "rug"
```

Because the model receives exactly the same input in both cases, it must produce exactly the same probability distribution.

After training it learned approximately:

```text
P("mat" | "sat on the") ≈ 0.497
P("rug" | "sat on the") ≈ 0.497
```

This is not a failure to memorize the training data.

It is the best kind of solution the model can represent given the information available in the context.

### Theoretical minimum loss

If all six unambiguous examples were predicted with probability approaching `1`, their loss would approach:

```text
0
```

For each ambiguous example, the optimal probability is approximately:

```text
0.5
```

so each contributes:

$$
-\log(0.5)=\log(2)\approx0.6931
$$

There are two such examples in a batch of eight, so the theoretical mean loss approaches:

$$
\frac{2\log(2)}{8}
\approx
0.1733
$$

Our final loss:

```text
0.1782
```

is already close to that floor.

This gives an important interpretation:

> A nonzero training loss does not necessarily mean optimization failed.

Some loss can represent uncertainty that cannot be resolved from the information supplied to the model.

This is fundamental to language modeling.

A model is not learning a deterministic lookup:

```text
context → one inevitable next word
```

It is learning a conditional probability distribution:

$$
P(\text{next token}\mid\text{context})
$$

## 32. Inspect the Full Learned Distribution for an Ambiguous Context

We have already seen that the context:

```text
["sat", "on", "the"]
```

appears twice in the training data with different targets:

```text
["sat", "on", "the"] → "mat"
["sat", "on", "the"] → "rug"
```

Because the input context is identical, the model must produce the same logits and therefore the same probability distribution for both examples.

We observed that the probability assigned to each target was approximately:

```text
P("mat" | "sat on the") ≈ 0.50
P("rug" | "sat on the") ≈ 0.50
```

But so far we have only inspected the probability assigned to the target token.

Now we will inspect the **entire learned probability distribution** over all vocabulary tokens.

This is important because a language model does not output a single word directly.

It outputs a distribution:

$$
P(\text{next token}\mid\text{context})
$$

over the entire vocabulary.

For our tiny vocabulary:

```text
V = 9
```

that means the model produces nine probabilities for this one context.

In [31]:
ambiguous_context_tokens = [
    "sat",
    "on",
    "the",
]

ambiguous_context_ids = torch.tensor(
    [
        token_to_id[token]
        for token in ambiguous_context_tokens
    ],
    dtype=torch.long,
)

print("Context tokens:")
print(ambiguous_context_tokens)

print("\nContext IDs:")
print(ambiguous_context_ids)

Context tokens:
['sat', 'on', 'the']

Context IDs:
tensor([7, 5, 8])


## 33. Run the Trained Model on This Context

Our trained model expects a batch of contexts with shape:

```text
[B, T]
```

But our single context currently has shape:

```text
[T]
```

So we add a batch dimension:

```text
[T]
↓
unsqueeze(0)
↓
[1, T]
```

Then we follow the same forward path used during training:

```text
token IDs
[1, T]

↓

embedding

[1, T, C]

↓

flatten

[1, T*C]

↓

prediction layer

[1, V]

↓

softmax

[1, V]
```

In [32]:
with torch.no_grad():
    ambiguous_context_batch = (
        ambiguous_context_ids.unsqueeze(0)
    )

    ambiguous_embeddings = embedding(
        ambiguous_context_batch
    )

    ambiguous_flattened = (
        ambiguous_embeddings.reshape(
            1,
            -1,
        )
    )

    ambiguous_logits = prediction_layer(
        ambiguous_flattened
    )

    ambiguous_probabilities = torch.softmax(
        ambiguous_logits,
        dim=-1,
    )

print(
    "Input shape:",
    ambiguous_context_batch.shape,
)

print(
    "Embedding shape:",
    ambiguous_embeddings.shape,
)

print(
    "Logits shape:",
    ambiguous_logits.shape,
)

print(
    "Probability shape:",
    ambiguous_probabilities.shape,
)

Input shape: torch.Size([1, 3])
Embedding shape: torch.Size([1, 3, 4])
Logits shape: torch.Size([1, 9])
Probability shape: torch.Size([1, 9])


In [33]:
print(
    "Next-token distribution for:",
    ambiguous_context_tokens,
)

print()

for token_id, probability in enumerate(
    ambiguous_probabilities[0]
):
    print(
        f"{id_to_token[token_id]:>5}: "
        f"{probability.item():.6f}"
    )

print(
    "\nProbability sum:",
    ambiguous_probabilities[0].sum().item(),
)

Next-token distribution for: ['sat', 'on', 'the']

  ate: 0.000684
  cat: 0.000318
  dog: 0.000520
 fish: 0.002131
  mat: 0.497040
   on: 0.001616
  rug: 0.496957
  sat: 0.000628
  the: 0.000106

Probability sum: 0.9999999403953552


## 34. Interpreting the Learned Distribution

For this context, we expect most of the probability mass to be concentrated on:

```text
"mat"
"rug"
```

because those are the two continuations observed in the training corpus.

Ideally, after sufficient training, the model should learn approximately:

```text
P("mat" | "sat on the") ≈ 0.5
P("rug" | "sat on the") ≈ 0.5
```

while assigning very little probability to unrelated tokens.

This demonstrates an important distinction.

The model does not learn:

```text
context → answer
```

It learns:

```text
context → distribution over possible next tokens
```

The highest-probability token is only one possible way to use that distribution.

Later, during generation, we can:

```text
choose the highest-probability token
```

or:

```text
sample from the distribution
```

Those choices produce different decoding behavior.

In [34]:
sorted_probabilities, sorted_ids = torch.sort(
    ambiguous_probabilities[0],
    descending=True,
)

print(
    "Ranked next-token predictions for:",
    ambiguous_context_tokens,
)

print()

for rank, (token_id, probability) in enumerate(
    zip(
        sorted_ids,
        sorted_probabilities,
    ),
    start=1,
):
    print(
        f"{rank:>2}. "
        f"{id_to_token[token_id.item()]:>5} "
        f"{probability.item():.6f}"
    )

Ranked next-token predictions for: ['sat', 'on', 'the']

 1.   mat 0.497040
 2.   rug 0.496957
 3.  fish 0.002131
 4.    on 0.001616
 5.   ate 0.000684
 6.   sat 0.000628
 7.   dog 0.000520
 8.   cat 0.000318
 9.   the 0.000106


## 35. From Training to Generation

We now have all the pieces needed to understand the basic interface of a language model.

Given a context:

```text
["sat", "on", "the"]
```

the trained model produces:

```text
logits over V tokens
↓
softmax
↓
probability distribution over V tokens
```

Training used the observed next token to compute:

```text
cross-entropy loss
```

But during generation there is no known target token.

Instead, we must choose a token from the model's predicted distribution.

That introduces a new problem:

> Given a probability distribution over the vocabulary, how should we choose the next token?

Several strategies are possible:

```text
greedy decoding
→ always choose the highest-probability token

sampling
→ randomly choose according to the learned probabilities

temperature
→ modify how sharp or flat the distribution is before sampling
```

Once a token is selected, it can be appended to the context:

```text
current context
↓
predict next-token distribution
↓
choose one token
↓
append it
↓
predict again
↓
repeat
```

That repeated process is **autoregressive generation**.

The next notebook will build this process directly.

## 36. Final Interpretation

The trained model learned the following distribution for:

```text
["sat", "on", "the"]
```

The two dominant predictions are:

```text
mat ≈ 0.4970
rug ≈ 0.4970
```

Together they receive approximately:

```text
99.4% of the total probability mass
```

while every other vocabulary token receives very little probability.

This matches the training data:

```text
["sat", "on", "the"] → "mat"
["sat", "on", "the"] → "rug"
```

Because the context is identical in both examples, the model cannot distinguish which occurrence it is processing.

It therefore learns approximately:

$$
P(\text{"mat"} \mid \text{"sat on the"})
\approx
0.5
$$

and:

$$
P(\text{"rug"} \mid \text{"sat on the"})
\approx
0.5
$$

This demonstrates that next-token prediction is not fundamentally:

```text
context
→ one stored answer
```

It is:

```text
context
→ probability distribution over the vocabulary
```

Training adjusts the model so that probability mass moves toward tokens that are supported by the observed data.

For unambiguous contexts, the model learned probabilities close to `1` for the observed continuation.

For the ambiguous context, it distributed probability across both observed continuations.

This also explains why the final training loss remained above zero.

The remaining loss is largely caused by uncertainty that cannot be resolved from the information contained in the context.

The complete training pipeline in this notebook was:

```text
context token IDs
[B, T]

↓

embedding lookup

[B, T, C]

↓

flatten context

[B, T*C]

↓

linear prediction layer

[B, V]

↓

logits

↓

softmax

probability distribution over vocabulary

↓

observed next-token target

↓

cross-entropy loss

↓

backpropagation

↓

gradients for prediction weights and embeddings

↓

optimizer update
```

The key conceptual result is:

> A language model learns a conditional probability distribution over the next token.

$$
P(\text{next token} \mid \text{context})
$$

During training, the observed next token tells us how to compute the loss.

During generation, there is no known target.

Instead, we must choose a token from this learned distribution.

That leads directly to autoregressive generation, greedy decoding, sampling, and temperature.